# Clase 5 · Laboratorio — Pipeline completo de limpieza sobre Saber 11

**Trabajo en parejas · 90 min.**

En la pre-clase viste cada técnica aislada. En este laboratorio aplicarás el **pipeline completo** al dataset real de Saber 11:

1. Diagnóstico
2. Corrección de texto
3. Imputación de nulos
4. Tratamiento de outliers
5. Normalización
6. Encoding + Transformación logarítmica

**Checkpoint del profesor a los 50 min** (después de la Tarea 3).

Al final producirás un `df_limpio` listo para modelar.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

df = pd.read_csv("datos/saber11_muestra_500k.csv")
df_original = df.copy()  # guardar copia para comparar al final
print(f"Datos crudos: {len(df):,} filas × {df.shape[1]} columnas")

Datos crudos: 500,000 filas × 22 columnas


## Tarea 1 (15 min) — Diagnóstico completo

Antes de tocar nada, hay que saber qué hay que arreglar.

1. Calcula el **porcentaje de nulos** de cada columna. Ordénalas de mayor a menor.
2. Cuenta las **filas duplicadas** completas.
3. Para las columnas de puntaje (`PUNT_*`), reporta **min, max, media** para identificar rangos sospechosos.
4. Para 3 columnas categóricas de tu elección, muestra los valores únicos con `value_counts()` y detecta si hay inconsistencias (espacios, mayúsculas raras).

Escribe 2-3 hallazgos en una celda Markdown al final.

In [5]:
# 1: porcentaje de nulos por columna (ordenado)
nulos_pct = (df.isna().sum() / len(df)) * 100
nulos_pct = nulos_pct.sort_values(ascending=False)
print("Top 15 columnas con más nulos:")
print(nulos_pct.head(15))

# 2: duplicados
n_dups = df.duplicated().sum()
print(f"\nFilas duplicadas: {n_dups:,}")

# 3: estadísticos de puntajes
puntajes = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
            "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES","PUNT_GLOBAL"]
print("\nRango de puntajes:")
print(df[puntajes].agg(["min", "max", "mean"]).T.round(2))

# 4: value_counts de 3 columnas categóricas a tu elección
cols_cat = ["COLE_NATURALEZA", "COLE_JORNADA", "FAMI_TIENEINTERNET"]
for col in cols_cat:
    print(f"\nValores únicos en {col}:")
    print(df[col].value_counts(dropna=False))
    print(f"Nulos en {col}: {df[col].isna().sum()}")
    print(f"¿Hay espacios o mayúsculas raras? {df[col].astype(str).str.contains(r'\s{2,}|[a-z]').any()}")


Top 15 columnas con más nulos:
FAMI_EDUCACIONPADRE         7.0144
FAMI_TIENEINTERNET          6.9666
FAMI_EDUCACIONMADRE         6.9324
PERIODO                     0.0000
COLE_DEPTO_UBICACION        0.0000
PUNT_GLOBAL                 0.0000
PUNT_INGLES                 0.0000
PUNT_SOCIALES_CIUDADANAS    0.0000
PUNT_C_NATURALES            0.0000
PUNT_MATEMATICAS            0.0000
PUNT_LECTURA_CRITICA        0.0000
COLE_MCPIO_UBICACION        0.0000
COLE_CALENDARIO             0.0000
COLE_BILINGUE               0.0000
ESTU_GENERO                 0.0000
dtype: float64

Filas duplicadas: 0

Rango de puntajes:
                          min    max    mean
PUNT_LECTURA_CRITICA      0.0  100.0   48.27
PUNT_MATEMATICAS          0.0  100.0   48.30
PUNT_C_NATURALES          0.0  100.0   48.30
PUNT_SOCIALES_CIUDADANAS  0.0  100.0   48.29
PUNT_INGLES               0.0  100.0   48.28
PUNT_GLOBAL               0.0  500.0  248.65

Valores únicos en COLE_NATURALEZA:
COLE_NATURALEZA
OFICIAL       359909


**Hallazgos del diagnóstico:**
1. El porcentaje de nulos está concentrado principalmente en variables socioeconómicas: `FAMI_EDUCACIONPADRE` (7.01%), `FAMI_TIENEINTERNET` (6.97%) y `FAMI_EDUCACIONMADRE` (6.93%). El resto de variables no presenta nulos relevantes.
2. No hay filas duplicadas completas: el dataset tiene 0 registros duplicados, así que no hay que eliminar filas por este motivo.
3. Los puntajes están en rangos esperados del 0 al 100 para cada prueba y de 0 a 500 para `PUNT_GLOBAL`, con medias alrededor de 48.3. En las columnas categóricas principales hay consistencia en `OFICIAL`/`NO OFICIAL` y `MAÑANA`/`TARDE`/`COMPLETA`, pero `FAMI_TIENEINTERNET` muestra valores `Si`/`No` con minúsculas y un 6.97% de nulos, por lo que necesita normalización y tratamiento posterior.


## Tarea 2 (20 min) — Corrección de texto + duplicados + imputación

1. **Corrige el texto** de todas las columnas categóricas de tipo string: `.str.strip().str.upper()`. Aplica a: COLE_NATURALEZA, COLE_JORNADA, COLE_CALENDARIO, COLE_BILINGUE, COLE_AREA_UBICACION, COLE_GENERO, FAMI_TIENEINTERNET, FAMI_TIENECOMPUTADOR, FAMI_TIENELAVADORA, FAMI_TIENEAUTOMOVIL, ESTU_GENERO.

2. **Elimina duplicados exactos** con `drop_duplicates()`. Reporta cuántas filas eliminaste.

3. **Imputa los nulos** eligiendo el método correcto:
   - Columnas numéricas continuas (PUNT_*) → mediana (son asimétricas)
   - Columnas categóricas (FAMI_EDUCACIONMADRE, FAMI_EDUCACIONPADRE, etc.) → moda

Justifica en Markdown por qué elegiste esos métodos.

In [7]:
# 1: normalizar texto de las 11 columnas listadas
cols_texto = ["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE",
              "COLE_AREA_UBICACION","COLE_GENERO","FAMI_TIENEINTERNET",
              "FAMI_TIENECOMPUTADOR","FAMI_TIENELAVADORA","FAMI_TIENEAUTOMOVIL",
              "ESTU_GENERO"]
for col in cols_texto:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().str.upper()

# 2: eliminar duplicados
n_antes = len(df)
df = df.drop_duplicates()
print(f"Duplicados eliminados: {n_antes - len(df):,}")

# 3a: imputar puntajes con la mediana
puntajes = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
            "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES","PUNT_GLOBAL"]
for col in puntajes:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# 3b: imputar categóricas con la moda
cols_categoricas = ["FAMI_EDUCACIONMADRE","FAMI_EDUCACIONPADRE","FAMI_ESTRATOVIVIENDA",
                    "FAMI_TIENEINTERNET","FAMI_TIENECOMPUTADOR"]
cols_categoricas = [col for col in cols_categoricas if col in df.columns]
for col in cols_categoricas:
    modo = df[col].mode()
    if not modo.empty:
        df[col] = df[col].fillna(modo.iloc[0])


# Validar que ya no quedan nulos en las columnas tratadas
print(f"\nNulos restantes en columnas tratadas: {df[puntajes + cols_categoricas].isna().sum().sum():,}")

Duplicados eliminados: 0

Nulos restantes en columnas tratadas: 0


**Justificación de los métodos elegidos:**
Usé la mediana para los puntajes porque las distribuciones de `PUNT_*` son asimétricas y la mediana es más robusta frente a valores extremos que la media. Para las variables categóricas usé la moda porque representa el valor más frecuente en la muestra y permite rellenar faltantes sin distorsionar demasiado la distribución del atributo. Además, al normalizar el texto a mayúsculas y sin espacios, evitamos inconsistencias típicas como `Si`, `si`, ` SI ` o valores con espacios extra.

## Tarea 3 (15 min) — Detección y tratamiento de outliers

1. Para PUNT_MATEMATICAS y PUNT_GLOBAL, detecta outliers con el método IQR.
2. Reporta cuántos outliers hay en cada columna.
3. Grafica un boxplot ANTES y DESPUÉS de eliminar los outliers de PUNT_MATEMATICAS.
4. **NO elimines los outliers todavía** — deja el DataFrame intacto. Solo etiquétalos.
5. Escribe en Markdown: ¿qué representan los outliers inferiores (puntajes muy bajos) en el contexto educativo colombiano? ¿Los eliminarías o los conservarías?

In [ ]:
def detectar_outliers_iqr(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    return (serie < Q1 - 1.5 * IQR) | (serie > Q3 + 1.5 * IQR)

# TODO 1: outliers en PUNT_MATEMATICAS y PUNT_GLOBAL
out_mat = ...
out_glob = ...
print(f"Outliers PUNT_MATEMATICAS: {out_mat.sum():,} ({out_mat.mean()*100:.2f}%)")
print(f"Outliers PUNT_GLOBAL:      {out_glob.sum():,} ({out_glob.mean()*100:.2f}%)")

# TODO 2: agregar columna is_outlier (SIN eliminar) para PUNT_MATEMATICAS
df["is_outlier_matematicas"] = ...

# TODO 3: boxplot antes vs después
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# ...
plt.tight_layout()
plt.show()

**¿Los eliminarías o los conservarías?**
_(justifica)_

## ⏸ Checkpoint del profesor (10 min)

Revisamos juntos:
- ¿Cuántos duplicados eliminaste en la Tarea 2?
- ¿Qué porcentaje de outliers tiene PUNT_MATEMATICAS?
- Un estudiante con puntaje = 0 en Matemáticas, ¿es outlier estadístico O es señal legítima de un problema educativo real?
- ¿Cuándo la limpieza podría HACER PERDER señal importante en los datos?

## Tarea 4 (15 min) — Normalización

1. Toma los 6 puntajes (PUNT_*) del DataFrame limpio (con duplicados eliminados e imputados).
2. Aplica los **3 métodos de normalización** (MinMax, Z-score, RobustScaler) a `PUNT_MATEMATICAS`. Crea 3 columnas nuevas.
3. Grafica 4 histogramas comparativos (original + los 3 normalizados) en una figura 1×4.
4. Aplica **Z-score** a los 6 puntajes y crea el DataFrame final `df_limpio` con las columnas normalizadas.
5. Guarda `df_limpio` como parquet en `../../datos/saber11_limpio.parquet` para usarlo en clases futuras.
6. Compara `df_original` vs `df_limpio` en una tabla: filas, nulos totales, media/std de PUNT_GLOBAL.

> Al terminar esta tarea NO guardes todavía `df_limpio` — lo guardarás después de la Tarea 5 con el encoding y log ya aplicados.

In [ ]:
# TODO 1: aplicar 3 escaladores a PUNT_MATEMATICAS
serie = df["PUNT_MATEMATICAS"].values.reshape(-1, 1)
df["MAT_MINMAX"] = MinMaxScaler().fit_transform(serie).ravel()
df["MAT_ZSCORE"] = StandardScaler().fit_transform(serie).ravel()
df["MAT_ROBUST"] = RobustScaler().fit_transform(serie).ravel()

# TODO 2: histograma comparativo 1x4
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
# ...
plt.tight_layout()
plt.show()

In [ ]:
# TODO: aplicar Z-score a los 6 puntajes
puntajes = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
            "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES","PUNT_GLOBAL"]
scaler = StandardScaler()
df_limpio = df.copy()
df_limpio[puntajes] = scaler.fit_transform(df[puntajes])

# TODO: guardar el resultado
df_limpio.to_parquet("../../datos/saber11_limpio.parquet", index=False)
print(f"df_limpio guardado: {df_limpio.shape}")

# TODO: comparación ORIGINAL vs LIMPIO
comparacion = pd.DataFrame({
    "Original": [len(df_original), df_original.isna().sum().sum(),
                 df_original["PUNT_GLOBAL"].mean(), df_original["PUNT_GLOBAL"].std()],
    "Limpio":   [len(df_limpio), df_limpio[puntajes + cols_categoricas].isna().sum().sum(),
                 df_original.loc[df_limpio.index, "PUNT_GLOBAL"].mean(),
                 df_original.loc[df_limpio.index, "PUNT_GLOBAL"].std()]
}, index=["Filas","Nulos totales","Media PUNT_GLOBAL","Std PUNT_GLOBAL"])
print(comparacion.round(2))

## Tarea 5 (15 min) — Encoding + Transformación logarítmica

En Machine Learning necesitas convertir texto a números y arreglar variables muy sesgadas. Este es el último paso antes de guardar `df_limpio`.

### Parte A — Encoding (8 min)

1. **Ordinal encoding** de `FAMI_ESTRATOVIVIENDA` — crea una nueva columna `ESTRATO_NUM` con el mapeo:
   `{"Sin Estrato": 0, "Estrato 1": 1, ..., "Estrato 6": 6}`

2. **One-Hot Encoding** de estas columnas nominales (usa `pd.get_dummies` con `drop_first=True`):
   `COLE_NATURALEZA`, `COLE_AREA_UBICACION`, `COLE_JORNADA`, `ESTU_GENERO`.

   Reporta cuántas columnas nuevas se generaron.

### Parte B — Transformación logarítmica (7 min)

3. Crea una nueva variable **agregada por municipio**: `estudiantes_por_mcpio = df.groupby("COLE_MCPIO_UBICACION").transform("size")`. Esa variable estará muy sesgada.

4. Aplica `np.log1p()` para crear la columna `log_estudiantes_mcpio`.

5. Grafica un histograma comparativo: original vs log-transformada. Reporta la asimetría antes y después.

In [ ]:
# TODO Parte A.1: Ordinal encoding de FAMI_ESTRATOVIVIENDA
mapeo_estrato = {
    "Sin Estrato": 0,
    "Estrato 1": 1, "Estrato 2": 2, "Estrato 3": 3,
    "Estrato 4": 4, "Estrato 5": 5, "Estrato 6": 6
}
df["ESTRATO_NUM"] = ...

# TODO Parte A.2: One-Hot Encoding con pd.get_dummies (drop_first=True)
cols_onehot = ["COLE_NATURALEZA", "COLE_AREA_UBICACION", "COLE_JORNADA", "ESTU_GENERO"]
n_cols_antes = df.shape[1]
df = ...
n_cols_despues = df.shape[1]
print(f"Columnas antes del OneHot: {n_cols_antes}")
print(f"Columnas después del OneHot: {n_cols_despues}")
print(f"Nuevas columnas creadas: {n_cols_despues - n_cols_antes}")

# Mostrar las nuevas columnas OneHot
nuevas = [c for c in df.columns if any(c.startswith(orig + "_") for orig in cols_onehot)]
print(f"\nColumnas OneHot generadas:")
for c in nuevas:
    print(f"  {c}")

In [ ]:
# TODO Parte B.3-4: crear estudiantes_por_mcpio y su log
df["estudiantes_por_mcpio"] = ...
df["log_estudiantes_mcpio"] = ...

# TODO Parte B.5: histograma comparativo
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ANTES
axes[0].hist(df["estudiantes_por_mcpio"], bins=60, color="coral", edgecolor="white")
axes[0].set_title(f"ANTES: asimetría = {df['estudiantes_por_mcpio'].skew():.2f}")
axes[0].set_xlabel("Estudiantes por municipio")

# DESPUÉS
axes[1].hist(df["log_estudiantes_mcpio"], bins=60, color="steelblue", edgecolor="white")
axes[1].set_title(f"DESPUÉS (log): asimetría = {df['log_estudiantes_mcpio'].skew():.2f}")
axes[1].set_xlabel("log(1 + estudiantes por municipio)")

plt.tight_layout()
plt.show()

print(f"\nAsimetría original: {df['estudiantes_por_mcpio'].skew():.3f}")
print(f"Asimetría con log:  {df['log_estudiantes_mcpio'].skew():.3f}")

### Reflexión Tarea 5

1. ¿Cuál columna original generó más columnas OneHot? ¿Por qué?
2. Después de aplicar `log1p`, ¿la asimetría se acercó a 0? ¿Qué implica eso para el modelado posterior?
3. Si tu dataset del proyecto tiene una columna categórica con **500 valores únicos**, ¿usarías OneHot? ¿Qué problema tendría?

_Respuesta:_

---

## Guardado final del DataFrame limpio

Ahora sí, guarda `df_limpio` con TODO aplicado: sin duplicados, imputado, normalizado, con encoding y transformaciones.

In [ ]:
# Guardar el DataFrame final
df.to_parquet("../../datos/saber11_limpio.parquet", index=False)
print(f"df_limpio guardado con {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nColumnas finales:")
for c in df.columns:
    print(f"  {c}")

### Reflexión final

1. ¿Qué diferencia notaste entre el DataFrame original y el limpio?
2. ¿En qué paso del pipeline hubo mayor pérdida de datos? ¿Se justifica?
3. Si tuvieras que aplicar este mismo pipeline a tu dataset del proyecto, ¿cuál paso te preocuparía más?

_Respuesta:_

---

## Entrega

Guarda este notebook con todos los TODO completados.

**Nombre:** `02_laboratorio_apellido1_apellido2.ipynb`
**Sube a:** Moodle → Clase 5 → Laboratorio.